# Exercise 1 — Protein Language Model playground

**27666 · June 2026 · Module 1**

**Goal**: take a real protein language model (ESM-2), get embeddings for three classes of sequences, see whether the embeddings separate the classes, and train a tiny classifier on them.

The three classes:
- **Nanobodies** (VHH single-domain antibodies) — the same nb1–nb10 used in Exercise 2
- **Human heavy-chain variable regions** (IGHV germlines) — a related but distinct antibody family
- **Shuffled controls** — same amino acid composition, no biological signal

We then go one step further: in **Section 8** we test whether the same embeddings can predict a quantitative biological property — the **optimal pH of an enzyme** — and compare against a sensible amino-acid-composition baseline. This is the closest match to the "frozen embeddings → linear probe → downstream task" pattern you will see throughout this morning.

**Runtime**: free-tier Colab CPU is fine (~10–15 min including the pH section). GPU is faster but not required.

## 🧑‍🎓 Student notebook — fill in the blanks!

This notebook contains **8 `TODO_N` markers** where you need to write a small piece of code before the cell will run. Each `TODO` is a real design choice a researcher makes — not a trick. If you get stuck, the markdown above the cell and the hint in the comment usually contain the answer.

**The blanks**:

- `TODO_1` (Sec. 4) — choice of pooling reduction in `embed_sequence`.
- `TODO_2` (Sec. 4) — slice to drop BOS and EOS tokens.
- `TODO_3` (Sec. 5) — UMAP `n_neighbors`.
- `TODO_4` (Sec. 6) — number of CV folds.
- `TODO_5` (Sec. 8) — maximum sequence length to embed.
- `TODO_6` (Sec. 8) — number of training sequences to embed.
- `TODO_7` (Sec. 8) — amino-acid composition formula.
- `TODO_8` (Sec. 8) — Ridge regularisation strength α.

Cells run top-to-bottom. If a cell crashes with `SyntaxError` or `NameError`, check for an unfilled `____` or `___`. A `_solved` companion notebook exists — try to resist looking at it!


## Attribution

- Model: **ESM-2** `facebook/esm2_t6_8M_UR50D` (Lin et al., *Science* 2023; via 🤗 `transformers`)
- Nanobody sequences nb1–nb10: `InstaNexus-main/json/sample_metadata.json` (Reverenna et al., DTU Biosustain/Bioengineering)
- Human IGHV germline sequences: IMGT/V-QUEST reference directory (public, IMGT)
- Notebook design inspiration: `ProteinLanguageWorkshop` (workshop) by Dimi Boeckaerts, `ESM2-Tutorial` (ProteinVision), `course_protein_language_modeling/notebooks/embeddings.ipynb` (Multiomics Analytics Group)

## 1. Setup

Run this cell once. On Colab the install takes ~1 minute the first time.

In [ ]:
# Colab: uncomment the next line if you get ImportError below.
# !pip install -q transformers==4.44.2 torch umap-learn scikit-learn matplotlib seaborn pandas numpy

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import AutoTokenizer, AutoModel

RNG_SEED = 42
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 2. Load the protein language model

We use **ESM-2 `t6_8M`** — the smallest ESM-2: 6 transformer layers, 8 M parameters, 320-dim embeddings. It runs on CPU in seconds per sequence.

The big sibling `t33_650M` (1280-dim, 650 M params) gives stronger embeddings but needs a GPU for batches. Try it later if you have time.

In [ ]:
MODEL_NAME = 'facebook/esm2_t6_8M_UR50D'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME}')
print(f'Parameters: {n_params/1e6:.1f} M')
print(f'Hidden dim: {model.config.hidden_size}')
print(f'Layers: {model.config.num_hidden_layers}')

## 3. Build the sequence dataset

### 3a. Nanobodies (VHH)

Ten nanobodies from `InstaNexus-main/json/sample_metadata.json`. We strip the C-terminal FLAG+His purification tag so we're embedding only the antibody domain.

In [ ]:
NANOBODIES_RAW = {
    'nb1':  'QVQLQESGGGLVQPGGSLRLSCAASGSIVNINYMRWYRQAPGKQRELVAVITSAGNTNYAESVKGRFTISIDNAKKMVFLQMNSLKPDDTAVYYCHADLRVRDGVRGDYWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb2':  'QVQLQESGGGLVQPGGSLRLSCAASGFTVSSVTLSWLRQAPGKGLEWVSDITSNGQTYYADSVKGRFTISRDNAKNTIYLQMNSLKADDSAVYFCAEDRWRSSNHPRGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb3':  'QVQLQESGGGLVQAGGSLRLSCLASGRTFSDYRIGWFRQAPGKEREFVSTIRNDDANTYYADSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCAAGARHTAQTMAAGKGIDYWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb4':  'QVQLQESGGGLVEPGGSLRLSCAASGFTFINYRMSWVRQAPGKGLEWVSGINPDGGTSYSDSVKGRFTISRDNAKNTLYLQMNSLKVEDTAVYYCIQSGTSRRGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb5':  'QVQLQESGGGLVQPGGSLRLSCAASGNIFSINYMKWYRQAPGKQRELVAVITDGGRTNYADSVKGRFAISRDNAKNTTYLQMSDLQPEDTAVYYCYADLRVVDGRHLPRGDYWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb6':  'QVQLQESGGGLVQPGGSLRLSCTASLNIFSINAMGWYRQAPGKQRELVAAITSGGSTNYADSVKGRFTISRDNAKSTVYLQMNSLKPEDTAVYYCHAEGPFNIATKEQYDYWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb7':  'QVQLQESGGGLVEPGGSLRLSCAVSGGSLNHYAMAWFRQAPGQEREGVACINRSGISTTYADSVKGRFTISRDNTKNTVWLQMNSLKPEDTGVYYCSAGKYYTVGHCDQDDYRGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb8':  'QVQLQESGGGLVQAGGSLRLSCAASGRTFDDYSMGWFRQAPGKEYEFVASINWSGSYTYYTDSVKGRFTISRDNAKNTVYLQMNSLKPDDTAVYYCAARDSIGVAVRRIDYDYWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb9':  'QVQLQESGGGLVQAGGSLRLSCAASGRTFSSYAMAWFRQAPGKEREFVASISWSGDSTYYADSVKGRFTISRDNAKNTWYLQMNSLKPEDTAVYYCNTEEESTGTYYEWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
    'nb10': 'QVQLQESGGGLVQPGGSLRLSCAASGSASSMYTLAWYRQAPGKQRELVALITSGHMTHYEDSVKGRFTISRDNAKEVLYLQMNSLKPEDTAVYFCNLHRLTSSDDDGRTWGQGTQVTVSSAAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH',
}

# Strip the C-terminal purification tag (AAA…HHHHHH).
# All nanobodies end with the VHH domain just before 'AAADYK'.
TAG_MARKER = 'AAADYKDHDGDYKDHDIDYKDDDDKGAAHHHHHH'
NANOBODIES = {name: seq.split('AAADYK')[0] for name, seq in NANOBODIES_RAW.items()}

for name, seq in NANOBODIES.items():
    print(f'{name}: {len(seq):3d} aa  {seq[:30]}…{seq[-10:]}')

### 3b. Human heavy-chain variable regions (IGHV germlines)

Ten human IGHV germline V-regions from IMGT — the human conventional heavy-chain repertoire. They share a fold with VHH nanobodies (immunoglobulin variable domain) but differ at framework-2 hallmarks and CDR length distribution.

In [ ]:
HUMAN_VH = {
    'IGHV1-2*02':  'EVQLVQSGAEVKKPGASVKVSCKASGYTFTSYAMHWVRQAPGQRLEWMGWINAGNGNTKYSQKFQGRVTITRDTSASTAYMELSSLRSEDTAVYYCAR',
    'IGHV1-3*01':  'QVQLVQSGAEVKKPGASVKVSCKASGYTFTGYYMHWVRQAPGQGLEWMGWINPNSGGTNYAQKFQGRVTMTRDTSISTAYMELSRLRSDDTAVYYCAR',
    'IGHV1-18*01': 'QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYGISWVRQAPGQGLEWMGWISAYNGNTNYAQKLQGRVTMTTDTSTSTAYMELRSLRSDDTAVYYCAR',
    'IGHV1-46*01': 'QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYYMHWVRQAPGQGLEWMGIINPSGGSTSYAQKFQGRVTMTRDTSTSTVYMELSSLRSEDTAVYYCAR',
    'IGHV3-7*01':  'EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYWMSWVRQAPGKGLEWVANIKQDGSEKYYVDSVKGRFTISRDNAKNSLYLQMNSLRAEDTAVYYCAR',
    'IGHV3-23*01': 'EVQLLESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSAISGSGGSTYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAK',
    'IGHV3-30*02': 'QVQLVESGGGVVQPGRSLRLSCAASGFTFSSYAMHWVRQAPGKGLEWVAVISYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR',
    'IGHV3-33*01': 'QVQLVESGGGVVQPGRSLRLSCAASGFTFSSYGMHWVRQAPGKGLEWVAVIWYDGSNKYYADSVKGRFTISRDNSKNTLYLQMNSLRAEDTAVYYCAR',
    'IGHV4-34*01': 'QVQLQQWGAGLLKPSETLSLTCAVYGGSFSGYYWSWIRQPPGKGLEWIGEINHSGSTNYNPSLKSRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR',
    'IGHV4-39*01': 'QLQLQESGPGLVKPSETLSLTCTVSGGSISSSSYYWGWIRQPPGKGLEWIGSIYYSGSTYYNPSLKSRVTISVDTSKNQFSLKLSSVTAADTAVYYCAR',
}

for name, seq in HUMAN_VH.items():
    print(f'{name:14s}: {len(seq):3d} aa')

### 3c. Shuffled controls

Take the nanobody amino-acid composition and shuffle the positions. Same residues, no biological signal. A well-trained pLM should see these as "not protein-like".

In [ ]:
rng = random.Random(RNG_SEED)

def shuffle_sequence(seq: str) -> str:
    chars = list(seq)
    rng.shuffle(chars)
    return ''.join(chars)

SHUFFLED = {
    f'shuf{i+1}': shuffle_sequence(seq)
    for i, seq in enumerate(NANOBODIES.values())
}

for name, seq in SHUFFLED.items():
    print(f'{name}: {len(seq):3d} aa  {seq[:40]}…')

In [ ]:
rows = (
    [{'name': n, 'class': 'nanobody (VHH)', 'sequence': s}      for n, s in NANOBODIES.items()] +
    [{'name': n, 'class': 'human VH',       'sequence': s}      for n, s in HUMAN_VH.items()]    +
    [{'name': n, 'class': 'shuffled',       'sequence': s}      for n, s in SHUFFLED.items()]
)
df = pd.DataFrame(rows)
df['length'] = df['sequence'].str.len()
print(df.groupby('class')['length'].describe())
df.head()

## 4. Tokenize & embed

For each sequence:
1. Tokenize into integer IDs (each amino acid → one token)
2. Forward pass through ESM-2 → per-residue contextual embeddings, shape `(seq_len, 320)`
3. **Mean-pool** over residues → a single 320-D vector per protein

Why mean-pooling: simplest aggregation, works surprisingly well. Alternatives: CLS-token embedding, max-pool, attention-weighted pool.

In [ ]:
@torch.no_grad()
def embed_sequence(seq: str) -> np.ndarray:
    """Mean-pooled per-protein embedding (shape: (hidden_dim,))."""
    inputs = tokenizer(seq, return_tensors='pt', add_special_tokens=True).to(DEVICE)
    out = model(**inputs)
    # last_hidden_state shape: (1, seq_len + 2, hidden_dim)  — incl. BOS/EOS
    hidden = out.last_hidden_state[0, _:_]    # TODO_2: ESM tokenises as [BOS] r1 r2 ... rL [EOS] — slice out the residues only (Python slice notation).
    return hidden.____(dim=0).cpu().numpy()    # TODO_1: which reduction collapses (seq_len, hidden) → (hidden,)? options: mean / sum / max / min — explain your choice.

In [ ]:
from tqdm.auto import tqdm

embeddings = np.stack([embed_sequence(s) for s in tqdm(df['sequence'], desc='embedding')])
print(f'embeddings shape: {embeddings.shape}  ({embeddings.shape[0]} sequences × {embeddings.shape[1]}-D)')

## 5. Visualise

Project the 320-D embeddings down to 2-D. Two complementary tools:
- **PCA**: linear, deterministic, interpretable ("what fraction of variance does PC1 explain?")
- **UMAP**: nonlinear, preserves local structure better, the default for visualising deep embeddings

In [ ]:
from sklearn.decomposition import PCA
import umap

pca = PCA(n_components=2, random_state=RNG_SEED)
pca_xy = pca.fit_transform(embeddings)
print(f'PC1 explained variance: {pca.explained_variance_ratio_[0]:.2%}')
print(f'PC2 explained variance: {pca.explained_variance_ratio_[1]:.2%}')

reducer = umap.UMAP(n_components=2, random_state=RNG_SEED, n_neighbors=___, min_dist=0.3)    # TODO_3: UMAP local neighbourhood size — small (~5) emphasises local clusters; large (~30) emphasises global structure. Pick a value for ~30 sequences.
umap_xy = reducer.fit_transform(embeddings)

In [ ]:
palette = {'nanobody (VHH)': '#d62728', 'human VH': '#1f77b4', 'shuffled': '#7f7f7f'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, xy, title in zip(axes, [pca_xy, umap_xy], ['PCA', 'UMAP']):
    for cls, color in palette.items():
        m = df['class'] == cls
        ax.scatter(xy[m, 0], xy[m, 1], c=color, label=cls, s=80, alpha=0.85, edgecolor='white', linewidth=0.6)
    ax.set_title(f'{title} of ESM-2 mean embeddings', fontsize=13)
    ax.set_xlabel(f'{title[0]}1')
    ax.set_ylabel(f'{title[0]}2')
    ax.legend(loc='best', frameon=False)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('ESM-2 t6_8M embeddings: nanobodies vs. human VH vs. shuffled', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('plm_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved 'plm_embeddings.png' — drop into Module 1 slide 16.")

**Stop and look at the plot.**

Questions:
- Do nanobodies and human VH separate?
- Are the shuffled controls clearly off to one side?
- Does PCA or UMAP give the cleaner picture? Why?
- Recall that nanobodies and human VH share the *same fold*. What is the model picking up on?

## 6. Tiny classifier on the embeddings

If the embeddings know what we hope they know, a linear classifier on top should already nail it. This is the simplest possible "frozen embeddings + linear probe" setup from Module 1 slide 17.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

X = embeddings
y = df['class'].values

clf = LogisticRegression(max_iter=2000, C=1.0, multi_class='multinomial')
cv = StratifiedKFold(n_splits=___, shuffle=True, random_state=RNG_SEED)    # TODO_4: number of cross-validation folds — with 10 examples per class, how many folds keep at least one example of each class per fold?
y_pred = cross_val_predict(clf, X, y, cv=cv)

print('5-fold cross-validated classification report:\n')
print(classification_report(y, y_pred, digits=3))

cm = confusion_matrix(y, y_pred, labels=list(palette.keys()))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=list(palette.keys()), yticklabels=list(palette.keys()), ax=ax)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title('Confusion matrix — 5-fold CV')
plt.tight_layout(); plt.show()

## 7. Bonus — zero-shot mutational scan on a nanobody CDR

Demonstrates the **third way to use a pLM** from Module 1 slide 17: take a wildtype sequence, mask one residue at a time, and read off the model's log-probability for each of the 20 amino acids.

We pick **nb6** (the same nanobody you'll assemble in Exercise 2) and scan a stretch around its CDR3 region. High log-probability means "the model thinks this residue belongs here" — a proxy for how well a mutant tolerates the substitution.

In [ ]:
from transformers import AutoModelForMaskedLM

mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()
MASK_ID = tokenizer.mask_token_id
AA_VOCAB = list('ACDEFGHIKLMNPQRSTVWY')
AA_TOKEN_IDS = {aa: tokenizer.convert_tokens_to_ids(aa) for aa in AA_VOCAB}

@torch.no_grad()
def scan_window(seq: str, start: int, end: int) -> pd.DataFrame:
    """For each position in [start, end), mask it and read log-probs for all 20 AAs."""
    rows = []
    base_ids = tokenizer(seq, return_tensors='pt').input_ids[0].tolist()
    for pos in range(start, end):
        ids = list(base_ids)
        ids[pos + 1] = MASK_ID  # +1 for BOS
        x = torch.tensor([ids]).to(DEVICE)
        logits = mlm(x).logits[0, pos + 1]
        logp = torch.log_softmax(logits, dim=-1).cpu().numpy()
        for aa in AA_VOCAB:
            rows.append({'position': pos, 'wt': seq[pos], 'aa': aa, 'log_prob': float(logp[AA_TOKEN_IDS[aa]])})
    return pd.DataFrame(rows)

In [ ]:
SCAN_TARGET = 'nb6'
nb_seq = NANOBODIES[SCAN_TARGET]
# CDR3 region is just before the WGQGTQ conserved motif
WGQ = nb_seq.find('WGQGTQ')
scan_start = max(0, WGQ - 22)
scan_end = WGQ
print(f'Scanning positions {scan_start}–{scan_end} (∼CDR3 region) of {SCAN_TARGET}, total {scan_end - scan_start} residues')
print(f'Wildtype stretch: {nb_seq[scan_start:scan_end]}')

scan_df = scan_window(nb_seq, scan_start, scan_end)
heat = scan_df.pivot(index='aa', columns='position', values='log_prob').loc[AA_VOCAB]

fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(heat, cmap='RdBu_r', center=heat.values.mean(), cbar_kws={'label': 'log P(aa | context)'}, ax=ax)
# Mark wildtype residues with a black dot
for pos in range(scan_start, scan_end):
    wt = nb_seq[pos]
    if wt in AA_VOCAB:
        ax.plot(pos - scan_start + 0.5, AA_VOCAB.index(wt) + 0.5, marker='o',
                color='black', markersize=4)
ax.set_xlabel(f'position in {SCAN_TARGET}')
ax.set_ylabel('amino acid')
ax.set_title(f'Zero-shot residue preferences from ESM-2 on {SCAN_TARGET} (black dots = wildtype)')
plt.tight_layout(); plt.show()

**Reading this plot**:
- A dark-red cell means the model thinks that amino acid is likely at that position
- Compare each black dot (wildtype) to the column maximum: is the wildtype already the model's favourite?
- Columns with a single hot row → conserved positions (likely framework)
- Columns with many hot rows → tolerant positions (likely CDR — where antibodies vary)

This is the same machinery that powers ESM-1v and AlphaMissense for variant effect prediction (Module 1 slide 17).

## 9. Discussion prompts

1. **Embedding visualisation (Section 5)**: Does ESM-2 separate nanobodies from human VH? They share a fold but differ at framework-2 hallmark residues (positions 37, 44, 45, 47 — Val/Phe/Glu/Gly conventional vs. Phe/Glu/Arg/Leu camelid). Could the model be picking those up?
2. **The shuffled control**: If shuffled sequences cluster separately, the model has learned "protein-like-ness". If they overlap with real proteins, your embedding is mostly amino-acid composition. Which case did you get?
3. **Classifier (Section 6)**: Where did it struggle? Look at the confusion matrix.
4. **Would AbLang (antibody-specific) do better?** Recall Module 1 slide 18 — Olson et al. 2022 showed AbLang separates V-gene families more cleanly than ESM-1b. Worth trying at home.
5. **Zero-shot scan (Section 7)**: which positions are the most tolerant? Do they align with the canonical CDR3 boundaries? Why might the first residue of CDR3 be highly conserved (hint: Cys at position 96 — disulfide-anchored start of CDR3)?
6. **pH prediction (Section 8)**: By how much did the ESM-2 embeddings beat the composition baseline? Did the embedding model still struggle at the extremes (pH < 3 or > 10)? Why?
7. **Foundation-model promise**: Across all four tasks (visualisation, classification, zero-shot scoring, regression), what is the *common* benefit of a pretrained pLM? What is the *common* failure mode?

Bring one observation to the closing discussion before we start Module 2.

In [ ]:
import urllib.request
from pathlib import Path

URLS = {
    'phopt_training.xlsx': 'https://github.com/ProteinVision/ESM2-Tutorial/raw/main/data/phopt_training.xlsx',
    'phopt_testing.xlsx':  'https://github.com/ProteinVision/ESM2-Tutorial/raw/main/data/phopt_testing.xlsx',
}
for fname, url in URLS.items():
    if not Path(fname).exists():
        urllib.request.urlretrieve(url, fname)
        print(f'downloaded {fname}')

train_full = pd.read_excel('phopt_training.xlsx')
test_full  = pd.read_excel('phopt_testing.xlsx')

# Cap sequence length AND subsample. ESM-2 t6_8M on CPU is fast but not magic — keep it sane.
MAX_LEN = ____     # TODO_5: cap sequence length to keep embedding fast on free Colab CPU. Trade-off: shorter = faster, but you discard real enzymes. Try 400 first.
N_TRAIN = ____     # TODO_6: how many training sequences to embed? More = better model, but slower. ~300 takes 4-5 min on CPU. ~50 is fast but poor.
N_TEST  = 100

train = train_full[train_full.sequence.str.len() < MAX_LEN].sample(n=N_TRAIN, random_state=RNG_SEED).reset_index(drop=True)
test  = test_full [test_full.sequence.str.len()  < MAX_LEN].sample(n=N_TEST,  random_state=RNG_SEED).reset_index(drop=True)

print(f'train: {len(train)} seqs, test: {len(test)} seqs')
print(f'pH range — train: [{train["optimal pH"].min():.1f}, {train["optimal pH"].max():.1f}]   test: [{test["optimal pH"].min():.1f}, {test["optimal pH"].max():.1f}]')

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(train['optimal pH'], bins=30, alpha=0.7, label='train', color='steelblue')
ax.hist(test['optimal pH'],  bins=30, alpha=0.7, label='test',  color='salmon')
ax.set(xlabel='optimal pH', ylabel='# enzymes', title='Label distribution')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
print('Embedding train sequences (this is the slow step)...')
X_train_emb = np.stack([embed_sequence(s) for s in tqdm(train.sequence)])

print('Embedding test sequences...')
X_test_emb = np.stack([embed_sequence(s) for s in tqdm(test.sequence)])

print(f'\nX_train_emb shape: {X_train_emb.shape}')
print(f'X_test_emb  shape: {X_test_emb.shape}')

In [ ]:
# Baseline feature: amino-acid composition vector — the simplest "bag of residues"
def aa_composition(seq: str) -> np.ndarray:
    L = len(seq)
    return np.array([___ for aa in AA_VOCAB])    # TODO_7: for each amino acid `aa` in AA_VOCAB, return its fractional occurrence in `seq` (count divided by length L).

X_train_comp = np.stack([aa_composition(s) for s in train.sequence])
X_test_comp  = np.stack([aa_composition(s) for s in test.sequence])

y_train = train['optimal pH'].values
y_test  = test['optimal pH'].values

print(f'Composition features:  train {X_train_comp.shape}, test {X_test_comp.shape}')
print(f'Embedding features:    train {X_train_emb.shape},  test {X_test_emb.shape}')

from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import spearmanr

RIDGE_ALPHA = ___    # TODO_8: Ridge regularisation strength. Small α = closer to ordinary least squares; large α = stronger shrinkage toward zero. Try 1.0.

results = []
for name, X_tr, X_te in [('AA composition (20-D)',     X_train_comp, X_test_comp),
                          ('ESM-2 embeddings (320-D)', X_train_emb,  X_test_emb)]:
    model = Ridge(alpha=RIDGE_ALPHA).fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    rho, _ = spearmanr(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    results.append({'features': name, 'R²': r2, 'Spearman ρ': rho, 'MSE': mse, 'y_pred': y_pred})
    print(f'{name:30s}  R²={r2:+.3f}  Spearman ρ={rho:+.3f}  MSE={mse:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, res in zip(axes, results):
    ax.scatter(y_test, res['y_pred'], alpha=0.6, edgecolor='white', s=40)
    lo, hi = 0, 13
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.4, label='y = x')
    ax.set(xlim=(lo, hi), ylim=(lo, hi),
           xlabel='true optimal pH', ylabel='predicted optimal pH',
           title=f"{res['features']}\nR² = {res['R²']:+.3f}   ρ = {res['Spearman ρ']:+.3f}")
    ax.legend(loc='lower right', frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.savefig('phopt_prediction.png', dpi=150, bbox_inches='tight'); plt.show()
print("Saved 'phopt_prediction.png'.")

**Reading the two panels:**

- The **dashed line** is `y = x`: perfect prediction would lie on it.
- A flat horizontal cloud means the model is predicting the dataset mean — i.e. not learning much.
- The **composition baseline** captures the fact that thermophilic / acidophilic enzymes have biased amino-acid content. It is not nothing.
- The **embedding model** should beat it: contextual embeddings carry information about local structure, hydrophobic packing and surface accessibility that pure composition cannot encode.

**Things to try (if time):**
- Bump `N_TRAIN` to 600 and re-run. How much does R² improve?
- Switch `Ridge` for `RandomForestRegressor`. Which features benefit more?
- Use the larger `ESM-2 t12_35M` model instead of `t6_8M` (1 line change). Worth it on CPU?
- Replace the regression head with `RidgeCV` to auto-pick `alpha`.

## 8. Discussion prompts

1. **Does ESM-2 separate nanobodies from human VH?** They share a fold but differ at the framework-2 hallmark residues (positions 37, 44, 45, 47 — Val/Phe/Glu/Gly conventional vs. Phe/Glu/Arg/Leu camelid). Could the model be picking those up?
2. **What does the shuffled control tell us?** If shuffled sequences cluster separately, the model has learned a notion of "protein-like-ness". If they overlap with real proteins, your embedding is mostly amino-acid composition.
3. **Where did our nanobody classifier struggle?** Look at the confusion matrix.
4. **Would AbLang (antibody-specific) do better?** Recall Module 1 slide 18 — Olson et al. 2022 showed AbLang separates V-gene families more cleanly than ESM-1b. Worth trying if you have time at home.
5. **Zero-shot scan**: which positions are the most tolerant in your scan? Do they align with the canonical CDR3 boundaries? Why might position 1 of CDR3 be highly conserved (hint: Cys at position 96 — the disulfide-anchored start of CDR3)?

## End of Exercise 1

After the break we move to **Module 2 — AI applications in biology**, ending with **Exercise 2** where you'll take real InstaNovo predictions of one of these same nanobodies (**nb6**) from a mass spectrometry experiment and assemble them back into the full protein with InstaNexus.